In [1]:
# Install dependencies and connect Google Drive
# Install the latest Hugging Face libraries
!pip install -q transformers datasets accelerate tokenizers

from google.colab import drive
import os

# Mount Drive to save files securely
drive.mount('/content/drive')

# Create the project directory in Drive if it does not exist
PROJECT_DIR = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"Working directory ready at: {PROJECT_DIR}")

Mounted at /content/drive
Working directory ready at: /content/drive/MyDrive/Colab_LLMs/Maia_Lite


In [2]:
# Import necessary libraries
from google.colab import userdata
import os
from huggingface_hub import login # Import the login function from Hugging Face Hub

# Load the Hugging Face token from Colab secrets
try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        # Attempt to log in with the Hugging Face token
        login(token=hf_token, add_to_git_credential=False) # 'add_to_git_credential=False' to not request git credentials
        print("Hugging Face token loaded and configured successfully via huggingface_hub.login().")
    else:
        print("WARNING: The secret 'HF_TOKEN' was found, but it is empty or None. Please check the value in Colab secrets.")
        # If the token is empty/None, still set the environment variable to an empty string to avoid future errors
        os.environ['HF_TOKEN'] = ''
except userdata.SecretNotFoundError:
    print("ATTENTION: The secret 'HF_TOKEN' was not found. Please add your Hugging Face token to Colab secrets.")
    os.environ['HF_TOKEN'] = '' # Ensure the environment variable is defined, even if empty
except Exception as e:
    print(f"An error occurred while loading or configuring the Hugging Face token: {e}")
    os.environ['HF_TOKEN'] = '' # Ensure the environment variable is defined, even if empty

Hugging Face token loaded and configured successfully via huggingface_hub.login().


In [3]:
from datasets import load_dataset, concatenate_datasets
from tokenizers import ByteLevelBPETokenizer
from transformers import GPT2TokenizerFast
import os
import shutil

PROJECT_DIR = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
tokenizer_dir = os.path.join(PROJECT_DIR, "tokenizador")

# Check if the tokenizer already exists and load it, otherwise train a new one.
if os.path.exists(tokenizer_dir) and os.path.isdir(tokenizer_dir) and \
   os.path.exists(os.path.join(tokenizer_dir, "vocab.json")) and \
   os.path.exists(os.path.join(tokenizer_dir, "merges.txt")):
    print(f"Tokenizer found at: {tokenizer_dir}. Loading existing tokenizer...")
    tokenizer = GPT2TokenizerFast.from_pretrained(tokenizer_dir, local_files_only=True)
    print("Tokenizer loaded successfully!")
else:
    print(f"Tokenizer not found or incomplete in {tokenizer_dir}. Training new tokenizer...")

    # 1. Load exact slices directly to Colab's local cache (WITHOUT streaming=True)
    # Proportionally divided to sum 50,000 articles in total (50% EN, 25% PT, 25% ES)
    print("Downloading Wikipedia slices to local memory (This takes about 1-2 minutes)...")
    wiki_en = load_dataset("wikimedia/wikipedia", "20231101.en", split="train[:25000]")
    wiki_pt = load_dataset("wikimedia/wikipedia", "20231101.pt", split="train[:12500]")
    wiki_es = load_dataset("wikimedia/wikipedia", "20231101.es", split="train[:12500]")

    # Join the locally downloaded datasets and shuffle in RAM
    print("Mixing and preparing the data...")
    mixed_dataset = concatenate_datasets([wiki_en, wiki_pt, wiki_es])
    mixed_dataset = mixed_dataset.shuffle(seed=42)

    # Generator used to feed the tokenizer trainer directly from RAM
    def extract_text():
        for item in mixed_dataset:
            yield item["text"]

    print("Training the tokenizer... This should now take 2 to 3 minutes.")
    raw_tokenizer = ByteLevelBPETokenizer()
    raw_tokenizer.train_from_iterator(
        extract_text(),
        vocab_size=50257, # Classic GPT-2 standard
        min_frequency=2,
        special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<mask>"]
    )

    # Save the tokenizer to Drive
    # REMOVE existing directory before recreating to force synchronization
    if os.path.exists(tokenizer_dir) and os.path.isdir(tokenizer_dir):
        print(f"Removing existing tokenizer directory: {tokenizer_dir}")
        shutil.rmtree(tokenizer_dir)

    os.makedirs(tokenizer_dir, exist_ok=True) # Create the directory again
    raw_tokenizer.save_model(tokenizer_dir)

    # Convert to the format usable by Hugging Face Trainer
    tokenizer = GPT2TokenizerFast.from_pretrained(tokenizer_dir, bos_token="<s>", eos_token="</s>", unk_token="<unk>", pad_token="<pad>", mask_token="<mask>")
    tokenizer.save_pretrained(tokenizer_dir)
    print(f"Tokenizer saved successfully at: {tokenizer_dir}")

Tokenizer found at: /content/drive/MyDrive/Colab_LLMs/Maia_Lite/tokenizador. Loading existing tokenizer...
Tokenizer loaded successfully!


In [ ]:
import os
import torch
import glob
from datasets import load_dataset, interleave_datasets
from transformers import GPT2LMHeadModel, GPT2TokenizerFast, TrainingArguments, Trainer, DataCollatorForLanguageModeling

# PyTorch memory optimization
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

PROJECT_DIR = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
tokenizer_dir = os.path.join(PROJECT_DIR, "tokenizador")
pretrained_checkpoints_dir = os.path.join(PROJECT_DIR, "checkpoints_pretreino") # Where to find pre-training
finetuning_output_dir = os.path.join(PROJECT_DIR, "checkpoints_finetuning")  # New folder for FT

# 1. Load Tokenizer and check GPU
tokenizer = GPT2TokenizerFast.from_pretrained(tokenizer_dir, local_files_only=True)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    device = torch.device("cuda")
else:
    gpu_name = "CPU"
    device = torch.device("cpu")
print(f"Running on: {gpu_name}")

# 2. Load the Pre-trained Model (Inherits the last pre-training checkpoint or the final model)
checkpoint_list = glob.glob(os.path.join(pretrained_checkpoints_dir, "checkpoint-*"))
if len(checkpoint_list) > 0:
    latest_checkpoint = max(checkpoint_list, key=os.path.getctime)
    print(f"Loading pre-training base from: {latest_checkpoint}")
    model = GPT2LMHeadModel.from_pretrained(latest_checkpoint)
else:
    final_model_path = os.path.join(PROJECT_DIR, "modelo_335M_final")
    if os.path.exists(final_model_path):
        print(f"Loading consolidated final model from: {final_model_path}")
        model = GPT2LMHeadModel.from_pretrained(final_model_path)
    else:
        raise FileNotFoundError("No pre-trained model found to start Fine-Tuning.")

model.to(device)
model.gradient_checkpointing_enable()

# 3. Load Conversation Datasets (Streaming)
ds_en = load_dataset("tatsu-lab/alpaca", split="train", streaming=True)                  # Original English
ds_pt = load_dataset("maritaca-ai/mira", split="train", streaming=True)                  # Portuguese
ds_es = load_dataset("bertin-project/alpaca-spanish", split="train", streaming=True)     # Spanish

# Interleave datasets maintaining a stable balance
conversation_dataset = interleave_datasets([ds_en, ds_pt, ds_es], probabilities=[0.4, 0.3, 0.3], seed=42)

# Prompt template to align conversations with GPT-2 standard
def map_conversation_prompt(example):
    instruction = example.get('instruction', '')
    context = example.get('input', '') if example.get('input') else ''
    response = example.get('output', '')

    # Assemble input by joining instruction and context
    user_input = f"{instruction}\n{context}".strip() if context else instruction

    # Classic prompt format for GPT-2 (Uses newlines and direct text)
    formatted_text = f"User: {user_input}\nAssistant: {response}{tokenizer.eos_token}"
    return {"text_formatted": formatted_text}

def tokenize_function(examples):
    return tokenizer(examples["text_formatted"], truncation=True, max_length=1024)

# Transform and tokenize, clearing old columns dynamically
mapped_dataset = conversation_dataset.map(map_conversation_prompt)
tokenized_dataset = mapped_dataset.map(tokenize_function, batched=True, remove_columns=list(conversation_dataset.features.keys()) + ["text_formatted"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# --- SPECIFIC CONFIGURATION FOR FINE-TUNING ON L4 ---
per_device_batch_size = 4
gradient_accumulation_steps = 8        # Global batch = 32
use_bf16 = True if "L4" in gpu_name or "A100" in gpu_name else False
use_fp16 = not use_bf16

# 4. Fine-Tuning Training Parameters
training_args = TrainingArguments(
    output_dir=finetuning_output_dir,
    max_steps=20000,               # Fine-tuning needs much fewer steps than pre-training
    per_device_train_batch_size=per_device_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    save_steps=1000,
    save_total_limit=2,
    logging_steps=50,
    bf16=use_bf16,
    fp16=use_fp16,
    learning_rate=5e-5,            # Much lower learning rate (e.g., 5e-5) to avoid destroying pre-training
    weight_decay=0.01,
    warmup_steps=1000,             # Warmup proportional to the smaller training size
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print(f"Starting FINE-TUNING on {gpu_name}...")
trainer.train()

# Save the final fine-tuned model for conversation
model.save_pretrained(os.path.join(PROJECT_DIR, "modelo_335M_chat_final"))
print("FINE-TUNING COMPLETED SUCCESSFULLY!")